In [0]:
%sql
CREATE OR REPLACE TABLE workspace.olist_gold_sql.dim_date AS
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key,
  d AS full_date,
  year(d) AS year,
  quarter(d) AS quarter,
  month(d) AS month,
  date_format(d, 'MMMM') AS month_name,
  dayofmonth(d) AS day_of_month,
  date_format(d, 'EEEE') AS day_name,
  dayofweek(d) IN (1, 7) AS is_weekend
FROM (
  SELECT explode(sequence(DATE'2016-01-01', DATE'2018-12-31', INTERVAL 1 DAY)) AS d
);

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.olist_gold_sql.dim_product AS
SELECT
  ROW_NUMBER() OVER (ORDER BY p.product_id) AS product_key,
  p.product_id,
  COALESCE(t.category_name_en, p.category_name_pt, 'unkown') AS category_name,
  p.weight_g, p.length_cm, p.height_cm, p.width_cm, p.photos_qty
FROM workspace.olist_silver_sql.products p
LEFT JOIN workspace.olist_silver_sql.category_translation t
  ON p.category_name_pt = t.category_name_pt;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.olist_gold_sql.dim_seller AS
SELECT
  ROW_NUMBER() OVER (ORDER BY seller_id) AS seller_key,
  seller_id, zip_prefix, city, state
FROM workspace.olist_silver_sql.sellers;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.olist_gold_sql.dim_customer AS
WITH cust_orders AS (
  SELECT c.customer_unique_id, c.customer_id, c.zip_prefix, c.city, c.state, o.purchased_at
  FROM workspace.olist_silver_sql.customers c
  JOIN workspace.olist_silver_sql.orders o ON o.customer_id = c.customer_id
),
flagged AS (
  SELECT *,
    CASE WHEN city = LAG(city) OVER (PARTITION BY customer_unique_id ORDER BY purchased_at, customer_id)
          AND state = LAG(state) OVER (PARTITION BY customer_unique_id ORDER BY purchased_at, customer_id)
         THEN 0 ELSE 1 END AS is_change
  FROM cust_orders
),
numbered AS (
  SELECT *,
    SUM(is_change) OVER (PARTITION BY customer_unique_id ORDER BY purchased_at, customer_id
                         ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS version_no
  FROM flagged
),
versions AS (
  SELECT customer_unique_id, version_no,
         MIN(zip_prefix) AS zip_prefix, MIN(city) AS city, MIN(state) AS state,
         MIN(purchased_at) AS valid_from
  FROM numbered
  GROUP BY customer_unique_id, version_no
)
SELECT
  ROW_NUMBER() OVER (ORDER BY customer_unique_id, version_no) AS customer_key,
  customer_unique_id, zip_prefix, city, state, valid_from,
  LEAD(valid_from) OVER (PARTITION BY customer_unique_id ORDER BY version_no) AS valid_to,
  (LEAD(valid_from) OVER (PARTITION BY customer_unique_id ORDER BY version_no)) IS NULL AS is_current
FROM versions;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.olist_gold_sql.fact_order_items AS
SELECT
  oi.order_id,
  oi.order_item_id,
  dc.customer_key,
  dp.product_key,
  ds.seller_key,
  CAST(date_format(o.purchased_at, 'yyyyMMdd') AS INT) AS purchase_date_key,
  CAST(date_format(o.delivered_at, 'yyyyMMdd') AS INT) AS delivered_date_key,
  o.order_status,
  oi.price,
  oi.freight_value,
  oi.price + oi.freight_value AS total_item_value,
  datediff(o.delivered_at, o.purchased_at) AS delivery_days,
  datediff(o.delivered_at, o.estimated_delivery_at) AS days_vs_estimate
FROM workspace.olist_silver_sql.order_items oi
JOIN workspace.olist_silver_sql.orders o ON o.order_id = oi.order_id
JOIN workspace.olist_silver_sql.customers c ON c.customer_id = o.customer_id
LEFT JOIN workspace.olist_gold_sql.dim_customer dc
  ON dc.customer_unique_id = c.customer_unique_id
 AND o.purchased_at >= dc.valid_from
 AND (dc.valid_to IS NULL OR o.purchased_at < dc.valid_to)
LEFT JOIN workspace.olist_gold_sql.dim_product dp ON dp.product_id = oi.product_id
LEFT JOIN workspace.olist_gold_sql.dim_seller ds ON ds.seller_id = oi.seller_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) AS fact_rows,
       COUNT(customer_key) AS with_customer,
       COUNT(product_key) AS with_product,
       COUNT(seller_key) AS with_seller
FROM workspace.olist_gold_sql.fact_order_items;

fact_rows,with_customer,with_product,with_seller
112650,112650,112650,112650


In [0]:
%sql
SELECT COUNT(*) AS version_rows,
       COUNT(DISTINCT customer_unique_id) AS people,
       SUM(CASE WHEN is_current THEN 1 ELSE 0 END) AS current_rows
FROM workspace.olist_gold_sql.dim_customer;

version_rows,people,current_rows
96220,96096,96096
